In [1]:
#第一轮清洗，筛掉无内容文件
from pathlib import Path
import re, json, hashlib
from collections import Counter
IN_DIR = Path(r"D:\shixi_agent\黄剑企业微信导出\1688858409408889")
OUT_DIR = Path(r"D:\shixi_agent\黄剑企业微信导出\2out")
MANIFEST = OUT_DIR / "manifest.jsonl"
DRY_RUN = False  # 先 True 看统计，确认后改 False
MIN_VALID_LINES = 3  # 至少多少行“真实文本”才保留（可调）
MIN_VALID_CHARS = 10  # 至少多少个字符才保留（可调）
# 这些基本都是“没用内容”
PLACEHOLDER_PAT = re.compile(
    r"^\s*(\[[^\]]+\]|@+|//@|转发|撤回了一条消息|你撤回了一条消息|加入了群聊|退出了群聊)\s*$"
)
# 常见无效：只有图片/文件/语音等占位
MEDIA_ONLY_PAT = re.compile(r"^\s*\[(图片|语音|视频|文件|动画表情|表情|位置|名片)\]\s*$")
# 你 sample 里这种 “我通过了你的联系人验证请求…” 也通常不算知识
SYSTEM_HINT_PAT = re.compile(r"联系人验证请求|现在我们可以开始聊天了|我通过了你的联系人验证请求")
# 一行消息的“头部”：昵称 + 时间（不强依赖格式，但尽量识别）
HEADER_PAT = re.compile(r"^(.+?)\s+(\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2})\s*$")

In [2]:
def try_read_text(p: Path):
    raw = p.read_bytes()
    # 先猜一轮：utf-8 / utf-8-sig / gb18030 / cp936 / latin1（最后兜底）
    candidates = ["utf-8", "utf-8-sig", "gb18030", "cp936", "latin1"]
    for enc in candidates:
        try:
            txt = raw.decode(enc)
            return txt, enc, None
        except UnicodeDecodeError as e:
            last = str(e)
    return "", "unknown", last

def fix_mojibake_if_needed(txt: str) -> str:
    # 典型特征：大量 "å¤§" 这种序列（UTF-8 被当 cp1252/latin1 再显示）
    # 处理：把当前字符串当 latin1 bytes，再按 utf-8 解回去
    if txt.count("å") + txt.count("ä") + txt.count("æ") + txt.count("ç") < 8:
        return txt
    try:
        return txt.encode("latin1", errors="ignore").decode("utf-8", errors="ignore")
    except Exception:
        return txt

In [3]:
def normalize_lines(txt: str):
    txt = txt.replace("\r\n", "\n").replace("\r", "\n")
    lines = [ln.strip() for ln in txt.split("\n")]
    # 去掉空行
    lines = [ln for ln in lines if ln]
    return lines

def clean_conversation(lines):
    cleaned = []
    valid_lines = []
    speakers = set()

    for ln in lines:
        # 去掉明显占位
        if PLACEHOLDER_PAT.match(ln): 
            continue
        if MEDIA_ONLY_PAT.match(ln):
            continue
        if ln == "@":
            continue

        m = HEADER_PAT.match(ln)
        if m:
            # 记录说话人，但 header 本身不算有效内容
            speakers.add(m.group(1).strip())
            cleaned.append(ln)
            continue

        # 去系统提示
        if SYSTEM_HINT_PAT.search(ln):
            cleaned.append(ln)  # 仍保留到输出里（可追溯），但不计有效
            continue

        cleaned.append(ln)

        # 统计有效内容：必须有汉字或数字/英文较多，且不是纯符号
        has_cjk = any("\u4e00" <= ch <= "\u9fff" for ch in ln)
        alpha_num = sum(ch.isalnum() for ch in ln)
        if has_cjk or alpha_num >= 6:
            # 再排除极短无意义
            if len(ln) >= 2:
                valid_lines.append(ln)

    valid_chars = sum(len(x) for x in valid_lines)
    return cleaned, valid_lines, valid_chars, speakers

def decide_keep(valid_lines, valid_chars):
    if len(valid_lines) < MIN_VALID_LINES:
        return False, f"valid_lines<{MIN_VALID_LINES}"
    if valid_chars < MIN_VALID_CHARS:
        return False, f"valid_chars<{MIN_VALID_CHARS}"
    # 纯“已通过验证/收到/好的”这种也过滤：有效词汇太少
    short_ack = sum(1 for x in valid_lines if x in {"收到", "好的", "ok", "OK", "嗯", "行"} or len(x) <= 2)
    if short_ack / max(1, len(valid_lines)) > 0.6:
        return False, "too_many_ack"
    return True, "ok"

In [4]:
def file_fingerprint(p: Path):
    b = p.read_bytes()
    return hashlib.md5(b).hexdigest()

def run():
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    stats = Counter()
    kept = 0
    total = 0

    manifest_rows = []

    for p in IN_DIR.rglob("*.txt"):
        total += 1
        txt, enc, err = try_read_text(p)
        txt2 = fix_mojibake_if_needed(txt)
        lines = normalize_lines(txt2)

        cleaned, valid_lines, valid_chars, speakers = clean_conversation(lines)
        keep, reason = decide_keep(valid_lines, valid_chars)

        stats["total"] += 1
        stats[f"reason:{reason}"] += 1
        stats[f"enc:{enc}"] += 1
        if err:
            stats["decode_error_count"] += 1

        out_path = OUT_DIR / p.name  # 同名写到 out（你也可改成保持子目录）
        row = {
            "file": str(p),
            "out_file": str(out_path),
            "keep": bool(keep),
            "reason": reason,
            "encoding": enc,
            "valid_lines": len(valid_lines),
            "valid_chars": int(valid_chars),
            "speakers_count": len(speakers),
            "md5": file_fingerprint(p),
        }
        manifest_rows.append(row)

        if keep:
            kept += 1
            if not DRY_RUN:
                out_text = "\n".join(cleaned).strip() + "\n"
                out_path.write_text(out_text, encoding="utf-8")

    # 写 manifest
    if not DRY_RUN:
        with MANIFEST.open("w", encoding="utf-8") as f:
            for r in manifest_rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print("DRY_RUN =", DRY_RUN)
    print("Total:", total, "Kept:", kept, "Dropped:", total - kept)
    print("Top reasons:")
    for k, v in stats.most_common(20):
        if k.startswith("reason:"):
            print(" ", k, v)

run()

DRY_RUN = False
Total: 1477 Kept: 530 Dropped: 947
Top reasons:
  reason:valid_lines<3 944
  reason:ok 530
  reason:too_many_ack 3


In [5]:
#第二轮筛选，剪切掉无意义内容
from pathlib import Path
import re, json
from collections import Counter

IN_DIR = Path(r"D:\shixi_agent\黄剑企业微信导出\2out")
OUT_DIR = Path(r"D:\shixi_agent\黄剑企业微信导出\2out2")
MANIFEST = OUT_DIR / "manifest2.jsonl"

DRY_RUN = False  # 先 True 看统计，确认后改 False

# 一行“header”：昵称 + 时间
HEADER_PAT = re.compile(r"^(.+?)\s+(\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2})\s*$")

# 无效占位（你这批里高频）
MEDIA_PAT = re.compile(r"^\s*\[(图片|语音|视频|文件|动画表情|表情|位置|名片)\]\s*$")

# 纯@、纯空白、或只剩@某人的“召唤”这类
AT_ONLY_PAT = re.compile(r"^\s*@\S+.*$")

# “收到/好的/ok”这类低信息 ACK（可按你数据再扩充）
ACK_SET = {
    "收到","好的","好","ok","OK","嗯","行","可以","已收到","👌","👍","1","2","3",
    "好的呢","好的呀","谢谢","谢了","辛苦了","辛苦"
}

# 系统提示（可按你数据再扩充）
SYSTEM_HINT_PAT = re.compile(r"联系人验证请求|现在我们可以开始聊天了|已成功注册|会话超时|帮助中心|为您找到相关问题")

In [6]:
def normalize_lines(text: str):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return [ln.strip() for ln in text.split("\n")]

def clean_lines(lines):
    """
    输入原始 lines（包含 header 与内容行）
    输出 cleaned_lines，并统计删除原因
    """
    cleaned = []
    drop = Counter()

    i = 0
    while i < len(lines):
        ln = lines[i].strip()

        if not ln:
            drop["empty_line"] += 1
            i += 1
            continue

        # header 行保留，但后面如果没有有效内容，会在后处理里删掉
        if HEADER_PAT.match(ln):
            cleaned.append(ln)
            i += 1
            continue

        # 内容行过滤
        if MEDIA_PAT.match(ln):
            drop["media_placeholder"] += 1
            i += 1
            continue

        if SYSTEM_HINT_PAT.search(ln):
            # 系统提示很多时候没知识价值：默认删
            drop["system_hint"] += 1
            i += 1
            continue

        if ln in ACK_SET or (len(ln) <= 2 and ln.lower() in {"ok"}):
            drop["ack"] += 1
            i += 1
            continue

        if ln == "@":
            drop["at_symbol"] += 1
            i += 1
            continue

        # 只@别人/群名，不包含实质内容：默认删
        if AT_ONLY_PAT.match(ln) and len(ln) <= 25:
            drop["at_only"] += 1
            i += 1
            continue

        cleaned.append(ln)
        i += 1

    # 后处理：删除“孤立 header”（header 后面没有内容、或下一个还是 header）
    final = []
    j = 0
    while j < len(cleaned):
        cur = cleaned[j]
        if HEADER_PAT.match(cur):
            nxt = cleaned[j+1] if j+1 < len(cleaned) else None
            if (nxt is None) or HEADER_PAT.match(nxt):
                drop["orphan_header"] += 1
                j += 1
                continue
        final.append(cur)
        j += 1

    return final, drop

In [7]:
def run():
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    stats = Counter()
    kept_files = 0
    total_files = 0

    manifest_rows = []

    for p in IN_DIR.rglob("*.txt"):
        total_files += 1
        text = p.read_text(encoding="utf-8", errors="ignore")
        raw_lines = normalize_lines(text)

        cleaned, drop = clean_lines(raw_lines)

        # 文件级保留条件：至少有 4 条内容（不含 header）或总字符一定量
        content_lines = [x for x in cleaned if not HEADER_PAT.match(x)]
        content_chars = sum(len(x) for x in content_lines)

        keep = (len(content_lines) >= 4 and content_chars >= 50)

        stats["files_total"] += 1
        stats["files_keep"] += int(keep)
        stats["files_drop"] += int(not keep)
        for k,v in drop.items():
            stats[f"drop:{k}"] += v

        out_path = OUT_DIR / p.name
        manifest_rows.append({
            "file": str(p),
            "out_file": str(out_path),
            "keep": bool(keep),
            "content_lines": len(content_lines),
            "content_chars": int(content_chars),
            "dropped_counts": dict(drop),
        })

        if keep:
            kept_files += 1
            if not DRY_RUN:
                out_text = "\n".join([ln for ln in cleaned if ln]).strip() + "\n"
                out_path.write_text(out_text, encoding="utf-8")

    if not DRY_RUN:
        with MANIFEST.open("w", encoding="utf-8") as f:
            for r in manifest_rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print("DRY_RUN =", DRY_RUN)
    print("Files:", total_files, "Kept:", kept_files, "Dropped:", total_files - kept_files)
    print("Top drops:")
    for k,v in stats.most_common(20):
        if k.startswith("drop:"):
            print(" ", k, v)

run()

DRY_RUN = False
Files: 530 Kept: 513 Dropped: 17
Top drops:
  drop:orphan_header 111452
  drop:ack 33361
  drop:at_only 20645
  drop:empty_line 530
  drop:system_hint 69


In [8]:
#第三清洗，整理成 知识库 JSONL（Q/A 记录），并做“安全级”脱敏（手机号+身份证）
from pathlib import Path
import re, json, hashlib
from collections import Counter

IN_DIR = Path(r"D:\shixi_agent\黄剑企业微信导出\2out2")
OUT_DIR = Path(r"D:\shixi_agent\黄剑企业微信导出\2out3")
OUT_JSONL = OUT_DIR / "kb.jsonl"
MANIFEST = OUT_DIR / "kb_manifest.jsonl"

DRY_RUN = False  # 先想看统计可改 True

# header 行：昵称 + 时间
HEADER_PAT = re.compile(r"^(.+?)\s+(\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2})\s*$")

# 判断“像问题”的规则（可逐步调）
QUESTION_PAT = re.compile(r"(？|\?|怎么|如何|为什么|为啥|能不能|是否|是不是|多少|几天|多久|怎么弄|怎么做|咋办)")

# 低信息回复（从 out2 已清过一轮，这里再兜底）
ACK_SET = {"收到","好的","好","ok","OK","嗯","行","可以","已收到","谢谢","辛苦","辛苦了"}

In [9]:
PHONE_PAT = re.compile(r"(?<!\d)(1[3-9]\d{9})(?!\d)")
IDCN_PAT = re.compile(r"(?<!\d)(\d{17}[\dXx])(?<!\d)")

def mask_text(s: str) -> str:
    if not s:
        return s
    s = PHONE_PAT.sub("<PHONE>", s)
    s = IDCN_PAT.sub("<ID>", s)
    return s

In [10]:
def read_lines(p: Path):
    text = p.read_text(encoding="utf-8", errors="ignore")
    # 去空行
    lines = [ln.strip() for ln in text.replace("\r\n","\n").replace("\r","\n").split("\n")]
    return [ln for ln in lines if ln]

def is_header(line: str):
    m = HEADER_PAT.match(line)
    if not m:
        return None
    return {"speaker": m.group(1).strip(), "ts": m.group(2).strip()}

def is_question(text: str):
    if not text or len(text) < 2:
        return False
    return bool(QUESTION_PAT.search(text))

def is_low_info(text: str):
    return (text in ACK_SET) or (len(text) <= 2)

def norm_for_hash(q: str, a: str):
    def norm(x):
        x = re.sub(r"\s+", " ", x).strip().lower()
        return x
    return norm(q), norm(a)

def qa_hash(q: str, a: str):
    q2, a2 = norm_for_hash(q, a)
    return hashlib.sha256((q2 + "\n" + a2).encode("utf-8")).hexdigest()[:24]

def extract_qa_from_file(p: Path, max_answer_lines=6, max_answer_chars=600):
    """
    返回 list[dict]：每个 dict 是一条 Q/A
    策略：
    - 只在“内容行”里找问题
    - 问题后面收集连续的非 header 内容当回答
    - 遇到下一条问题或下一条 header 后面空等情况会停止
    """
    lines = read_lines(p)

    # 把 lines 转成事件流：header 或 content
    events = []
    cur = {"speaker": None, "ts": None}
    for ln in lines:
        h = is_header(ln)
        if h:
            cur = h
            events.append({"type":"header", **h})
        else:
            events.append({"type":"content", "speaker": cur["speaker"], "ts": cur["ts"], "text": ln})

    out = []
    i = 0
    while i < len(events):
        e = events[i]
        if e["type"] != "content":
            i += 1
            continue

        q = e["text"].strip()
        if not is_question(q):
            i += 1
            continue

        # 收集回答：从下一条开始抓若干条 content，直到遇到下一条“看起来是问题”的 content 或抓够
        ans_lines = []
        ans_speakers = []
        j = i + 1
        while j < len(events) and len(ans_lines) < max_answer_lines and sum(len(x) for x in ans_lines) < max_answer_chars:
            e2 = events[j]
            if e2["type"] != "content":
                j += 1
                continue
            t = e2["text"].strip()
            if not t:
                j += 1
                continue
            if is_question(t) and len(ans_lines) >= 1:
                break
            if is_low_info(t):
                j += 1
                continue
            ans_lines.append(t)
            ans_speakers.append(e2.get("speaker"))
            j += 1

        answer = "\n".join(ans_lines).strip()

        # 最小质量门槛
        if len(answer) >= 8:
            q_m = mask_text(q)
            a_m = mask_text(answer)
            out.append({
                "question": q_m,
                "answer": a_m,
                "source_file": str(p),
                "q_speaker": e.get("speaker"),
                "q_ts": e.get("ts"),
                "answer_speakers": [s for s in ans_speakers if s],
            })

        i += 1

    return out

In [11]:
def run():
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    stats = Counter()
    seen = set()
    kept = 0

    all_rows = []
    for p in IN_DIR.rglob("*.txt"):
        rows = extract_qa_from_file(p)
        stats["files"] += 1
        stats["qa_raw"] += len(rows)

        for r in rows:
            h = qa_hash(r["question"], r["answer"])
            if h in seen:
                stats["qa_dup_drop"] += 1
                continue
            seen.add(h)
            r2 = {
                "question": r["question"],
                "answer": r["answer"],
                "source_file": r["source_file"],
                "q_speaker": r["q_speaker"],
                "q_ts": r["q_ts"],
                "hash": h,
            }
            all_rows.append(r2)
            kept += 1

    stats["qa_kept"] = kept

    print("DRY_RUN =", DRY_RUN)
    print("Files:", stats["files"])
    print("Q/A raw:", stats["qa_raw"])
    print("Q/A kept:", stats["qa_kept"])
    print("Q/A dropped dup:", stats["qa_dup_drop"])

    if DRY_RUN:
        # 抽样展示 5 条看看质量
        for sample in all_rows[:5]:
            print("\n---")
            print("Q:", sample["question"])
            print("A:", sample["answer"][:200])
        return

    with OUT_JSONL.open("w", encoding="utf-8") as f:
        for r in all_rows:
            f.write(json.dumps({"question": r["question"], "answer": r["answer"]}, ensure_ascii=False) + "\n")

    with MANIFEST.open("w", encoding="utf-8") as f:
        for r in all_rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print("Wrote:", OUT_JSONL)
    print("Manifest:", MANIFEST)

run()

DRY_RUN = False
Files: 513
Q/A raw: 24579
Q/A kept: 24485
Q/A dropped dup: 94
Wrote: D:\shixi_agent\黄剑企业微信导出\2out3\kb.jsonl
Manifest: D:\shixi_agent\黄剑企业微信导出\2out3\kb_manifest.jsonl


In [12]:
#第四轮清洗
from pathlib import Path
import json, re, hashlib
from collections import Counter, defaultdict

IN_KB = Path(r"D:\shixi_agent\黄剑企业微信导出\2out3\kb.jsonl")
OUT_DIR = Path(r"D:\shixi_agent\黄剑企业微信导出\2out4")

OUT_KB = OUT_DIR / "kb_clean.jsonl"
OUT_DROP = OUT_DIR / "kb_drop.jsonl"
OUT_STATS = OUT_DIR / "kb_stats.json"
OUT_LEAKS = OUT_DIR / "kb_leaks_found.jsonl"  # 记录“发现并已脱敏”的样本位置（不泄露原文）

# 去重策略：按“规范化后的 question”去重，保留信息更丰富的一条
DEDUP_BY = "question"  # 固定用 question 去重（你也可改为 "qa" 做更严格精确去重）

# answer 里这些占位/分隔符统统清掉
MEDIA_TOKENS = [
    r"\[图片\]", r"\[语音\]", r"\[视频\]", r"\[文件\]", r"\[表情\]", r"\[动画表情\]",
    r"\[位置\]", r"\[名片\]"
]
SEPARATOR_TOKENS = [r"^-{3,}$", r"^_{3,}$", r"^={3,}$", r"^—{3,}$", r"^~{3,}$", r"^·{3,}$"]

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [13]:
_ws_re = re.compile(r"[ \t\u3000]+")
_multi_nl = re.compile(r"\n{3,}")

media_re = re.compile("|".join([re.escape(x) for x in ["[图片]","[语音]","[视频]","[文件]","[表情]","[动画表情]","[位置]","[名片]"]]))
# 也处理被引号包住的："[图片]" 或 “【图片】” 等
media_quoted_re = re.compile(r"([\"“”'‘’]|\u300a|\u300b|\u3010|\u3011|\u300c|\u300d)?\s*\[(图片|语音|视频|文件|表情|动画表情|位置|名片)\]\s*([\"“”'‘’])?")

sep_line_re = re.compile("|".join(SEPARATOR_TOKENS))

def norm_question(q: str) -> str:
    if not q:
        return ""
    q = q.strip()
    q = _ws_re.sub(" ", q)
    q = q.replace("？", "?").replace("﹖", "?")
    q = q.replace("。。", "。")
    q = q.strip()
    return q

def clean_answer(a: str) -> str:
    if not a:
        return ""
    a = a.replace("\r\n", "\n").replace("\r", "\n")
    # 先删媒体占位（含引号包住的）
    a = media_quoted_re.sub("", a)
    a = media_re.sub("", a)

    # 去掉一些“纯分隔符行”
    lines = []
    for ln in a.split("\n"):
        s = ln.strip()
        if not s:
            continue
        if sep_line_re.match(s):
            continue
        lines.append(s)

    a2 = "\n".join(lines)
    a2 = _multi_nl.sub("\n\n", a2).strip()
    return a2

In [14]:
# 手机号：支持 1xx xxxx xxxx / 1xx-xxxx-xxxx / 连续11位
phone_re = re.compile(r"""
(?<!\d)
(1[3-9]\d)
[\s\-]?
(\d{4})
[\s\-]?
(\d{4})
(?!\d)
""", re.X)

# 身份证：18位，最后一位可 X/x；支持中间夹空格/短横线
id18_re = re.compile(r"""
(?<![0-9Xx])
(\d{6})
[\s\-]?
(\d{4})
[\s\-]?
(\d{2})
[\s\-]?
(\d{2})
[\s\-]?
(\d{3})
([0-9Xx])
(?![0-9Xx])
""", re.X)

def mask_sensitive(text: str):
    """
    返回：masked_text, leaks(list[str])
    leaks 只记录命中类型，不回填原始敏感数字（避免二次泄露）
    """
    if not text:
        return "", []

    leaks = []

    def _phone_sub(m):
        leaks.append("PHONE")
        return "<PHONE>"

    def _id_sub(m):
        leaks.append("ID18")
        return "<ID>"

    t = text
    t2 = phone_re.sub(_phone_sub, t)
    t3 = id18_re.sub(_id_sub, t2)

    # 去重 leaks
    if leaks:
        leaks = sorted(set(leaks))
    return t3, leaks

In [15]:
def record_key(rec):
    q = norm_question(rec.get("question",""))
    a = rec.get("answer","") or ""
    a = clean_answer(a)
    if DEDUP_BY == "qa":
        return hashlib.md5((q + "\n" + a).encode("utf-8")).hexdigest()
    return q  # by question

def score_record(rec):
    """
    去重冲突时选“更有信息”的那条：
    - answer 字符更多
    - 行数更多
    """
    a = rec.get("answer","") or ""
    a = clean_answer(a)
    return (len(a), a.count("\n"))

stats = Counter()
kept = {}
dropped = []

leaks_rows = []

with IN_KB.open("r", encoding="utf-8", errors="ignore") as f:
    for idx, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except Exception:
            stats["bad_json"] += 1
            continue

        stats["in_total"] += 1

        q0 = rec.get("question","") or ""
        a0 = rec.get("answer","") or ""

        q = norm_question(q0)
        a = clean_answer(a0)

        # 脱敏补漏（同时对 question/answer 做）
        q2, leaks_q = mask_sensitive(q)
        a2, leaks_a = mask_sensitive(a)
        leaks = sorted(set(leaks_q + leaks_a))

        if leaks:
            stats["masked_records"] += 1
            leaks_rows.append({
                "line_no": idx,
                "types": leaks,
                "hash": rec.get("hash",""),
                "source_file": rec.get("source_file",""),
            })

        rec["question"] = q2
        rec["answer"] = a2

        # 如果 answer 清完变空，仍保留（你没要求过滤），但计数
        if not rec["answer"].strip():
            stats["empty_answer_after_clean"] += 1

        k = record_key(rec)
        if not k:
            stats["empty_key"] += 1
            dropped.append({"why":"empty_key", "line_no": idx, "hash": rec.get("hash",""), "source_file": rec.get("source_file","")})
            continue

        if k not in kept:
            kept[k] = rec
            stats["kept"] += 1
        else:
            # 去重：保留“信息更丰富”的那条
            old = kept[k]
            if score_record(rec) > score_record(old):
                kept[k] = rec
                dropped.append({"why":"dedup_replaced", "line_no": idx, "hash": rec.get("hash",""), "source_file": rec.get("source_file","")})
                stats["dedup_replaced"] += 1
            else:
                dropped.append({"why":"dedup_dropped", "line_no": idx, "hash": rec.get("hash",""), "source_file": rec.get("source_file","")})
                stats["dedup_dropped"] += 1

stats["out_total"] = len(kept)

# 写出
with OUT_KB.open("w", encoding="utf-8") as w:
    for rec in kept.values():
        w.write(json.dumps(rec, ensure_ascii=False) + "\n")

with OUT_DROP.open("w", encoding="utf-8") as w:
    for rec in dropped:
        w.write(json.dumps(rec, ensure_ascii=False) + "\n")

with OUT_STATS.open("w", encoding="utf-8") as w:
    w.write(json.dumps(stats, ensure_ascii=False, indent=2))

with OUT_LEAKS.open("w", encoding="utf-8") as w:
    for r in leaks_rows:
        w.write(json.dumps(r, ensure_ascii=False) + "\n")

print("输入条数:", stats["in_total"])
print("输出条数:", stats["out_total"])
print("去重丢弃:", stats["dedup_dropped"], "替换:", stats["dedup_replaced"])
print("脱敏命中记录数:", stats["masked_records"])
print("空answer(清洗后):", stats["empty_answer_after_clean"])
print("输出目录:", OUT_DIR)

输入条数: 24485
输出条数: 23507
去重丢弃: 651 替换: 327
脱敏命中记录数: 1008
空answer(清洗后): 1
输出目录: D:\shixi_agent\黄剑企业微信导出\2out4


In [16]:
#第五轮清洗，数据脱敏、查找问题
from pathlib import Path
import json, re

IN_KB = Path(r"D:\shixi_agent\黄剑企业微信导出\2out4\kb_clean.jsonl")
OUT_LEAKS = Path(r"D:\shixi_agent\黄剑企业微信导出\2out4\leaks_remaining.jsonl")

# 手机号：支持 1xx xxxx xxxx / 1xx-xxxx-xxxx / 连续11位
phone_re = re.compile(r"(?<!\d)(1[3-9]\d)[\s\-]?(\d{4})[\s\-]?(\d{4})(?!\d)")

# 身份证：18位，最后一位可 X/x；支持中间夹空格/短横线
id18_re = re.compile(r"(?<![0-9Xx])(\d{6})[\s\-]?(\d{4})[\s\-]?(\d{2})[\s\-]?(\d{2})[\s\-]?(\d{3})([0-9Xx])(?![0-9Xx])")

def mask_preview(s: str, m: re.Match, keep=3):
    # 打码展示（避免二次泄露）
    start, end = m.span()
    hit = s[start:end]
    if len(hit) <= keep*2:
        return "<MASKED>"
    return hit[:keep] + "*"*(len(hit)-keep*2) + hit[-keep:]

hits = 0
with IN_KB.open("r", encoding="utf-8", errors="ignore") as f, OUT_LEAKS.open("w", encoding="utf-8") as w:
    for i, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except Exception:
            continue

        q = rec.get("question","") or ""
        a = rec.get("answer","") or ""

        found = []

        for m in phone_re.finditer(q):
            found.append({"type":"PHONE", "field":"question", "preview": mask_preview(q, m)})
        for m in phone_re.finditer(a):
            found.append({"type":"PHONE", "field":"answer", "preview": mask_preview(a, m)})

        for m in id18_re.finditer(q):
            found.append({"type":"ID18", "field":"question", "preview": mask_preview(q, m)})
        for m in id18_re.finditer(a):
            found.append({"type":"ID18", "field":"answer", "preview": mask_preview(a, m)})

        if found:
            hits += 1
            w.write(json.dumps({"line_no": i, "found": found}, ensure_ascii=False) + "\n")

print("扫描完成。疑似漏网记录数 =", hits)
print("输出：", OUT_LEAKS)

扫描完成。疑似漏网记录数 = 0
输出： D:\shixi_agent\黄剑企业微信导出\2out4\leaks_remaining.jsonl


In [17]:
from pathlib import Path
import json, re
import random

IN_KB = Path(r"D:\shixi_agent\黄剑企业微信导出\out4\kb_clean.jsonl")
OUT_ISSUES = Path(r"D:\shixi_agent\黄剑企业微信导出\out4\quality_issues.jsonl")

media_pat = re.compile(r"\[(图片|语音|视频|文件|表情|动画表情|位置|名片)\]")
sep_pat = re.compile(r"(^-{3,}$|^_{3,}$|^={3,}$|^—{3,}$)", re.M)

def issues_for(rec):
    q = (rec.get("question") or "").strip()
    a = (rec.get("answer") or "").strip()
    issues = []
    if len(q) < 4:
        issues.append("q_too_short")
    if len(a) < 15:
        issues.append("a_too_short")
    if media_pat.search(a) or media_pat.search(q):
        issues.append("media_placeholder_left")
    if sep_pat.search(a):
        issues.append("separator_left")
    return issues

issues_count = 0
sample = []

with IN_KB.open("r", encoding="utf-8", errors="ignore") as f, OUT_ISSUES.open("w", encoding="utf-8") as w:
    for i, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except Exception:
            continue

        iss = issues_for(rec)
        if iss:
            issues_count += 1
            row = {
                "line_no": i,
                "issues": iss,
                "question": (rec.get("question") or "")[:120],
                "answer_preview": (rec.get("answer") or "")[:200],
            }
            w.write(json.dumps(row, ensure_ascii=False) + "\n")
            if len(sample) < 30 and random.random() < 0.02:
                sample.append(row)

print("发现疑似问题条目数 =", issues_count)
print("输出：", OUT_ISSUES)
print("随机样本(最多30条)：")
for r in sample[:10]:
    print("-", r["issues"], r["question"])

发现疑似问题条目数 = 1062
输出： D:\shixi_agent\黄剑企业微信导出\out4\quality_issues.jsonl
随机样本(最多30条)：
- ['a_too_short'] 嗯，我这边让做了一个跟咱们一样的报价啊，保费的话呢，我还在沟通，不知道能不能降，但是我们把那个免赔额免赔给降低了。
- ['a_too_short'] 我和长凌通个电话，看看怎么弄
- ['q_too_short'] 年单?
- ['a_too_short'] 我就问一下你们要是在海淀那个位置，我直接过去，可能是，还是关键的操作，哎呀，我好累呀，我今天一天都在搞这个东西，一直搞不通，所以说你能不能发个定位，要不然我过去过去你给我指点一下，有可能就是一下子都搞清楚了，现在你怎么说我也好累呀，我确实搞
- ['a_too_short'] 门诊治疗还是住院治疗?
- ['a_too_short'] 高忠升 <ID>；杨郎杰 <ID> 名字打错了，这种情况需要改下名字的话，怎么改 @助理
- ['a_too_short'] 替换选其他变更么?
- ['a_too_short'] 员工名字叫什么?
- ['a_too_short'] 电工是普通的电工吗?
- ['a_too_short'] 如果只重新增员，那个信息是不是不能修改呀


In [18]:
#第六轮清洗，合并
from pathlib import Path
import json, re
from collections import Counter

IN = Path(r"D:\shixi_agent\黄剑企业微信导出\2out3\kb_manifest.jsonl")
OUT_DIR = Path(r"D:\shixi_agent\黄剑企业微信导出\2out5")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_KB = OUT_DIR / "kb_merged.jsonl"
OUT_STATS = OUT_DIR / "merge_stats.json"
OUT_EX = OUT_DIR / "merge_examples.jsonl"

In [19]:
MIN_ANSWER_LEN = 0      # 希望“合并后的 answer”至少这么长
MAX_APPEND = 4           # 最多往后拼接几条
MAX_GAP_SEC = 15 * 60    # 超过这个时间间隔就不拼了（默认15分钟）

# 判断“下一条 question 是不是新问题”的启发式：含问号/疑问词就算新问题
new_q_pat = re.compile(r"[?？]|怎么|如何|为什么|是否|能不能|可不可以|多少|几点|哪里|谁|要不要")

def is_new_question(q: str) -> bool:
    if not q: 
        return False
    return bool(new_q_pat.search(q))

def parse_ts(ts: str):
    # ts 形如 "2023-05-25 11:48:11"
    import datetime
    try:
        return datetime.datetime.strptime(ts, "%Y-%m-%d %H:%M:%S")
    except Exception:
        return None

def clean_text(s: str) -> str:
    if not s:
        return ""
    s = s.replace("\r\n", "\n").replace("\r", "\n").strip()
    # 先把中文引号/特殊引号统一成普通引号，避免复制导致的语法/编码问题
    s = s.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
    # 再清理媒体占位（含被引号包起来的）
    s = re.sub(r'\s*["\']?\[(图片|语音|视频|文件|表情|动画表情|位置|名片)\]["\']?\s*', "", s)
    # 合并多余空行
    s = re.sub(r"\n{3,}", "\n\n", s).strip()
    return s

In [20]:
rows_by_file = {}
stats = Counter()

with IN.open("r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        rec["question"] = clean_text(rec.get("question",""))
        rec["answer"] = clean_text(rec.get("answer",""))
        sf = rec.get("source_file","") or ""
        rows_by_file.setdefault(sf, []).append(rec)
        stats["in_total"] += 1

# 排序
for sf, arr in rows_by_file.items():
    arr.sort(key=lambda r: (r.get("q_ts",""), r.get("hash","")))

In [21]:
import random
examples_written = 0

with OUT_KB.open("w", encoding="utf-8") as out, OUT_EX.open("w", encoding="utf-8") as ex:
    for sf, arr in rows_by_file.items():
        i = 0
        while i < len(arr):
            cur = dict(arr[i])  # copy
            q = cur.get("question","")
            a = cur.get("answer","")
            cur_ts = parse_ts(cur.get("q_ts","") or "")

            # 如果 answer 太短，尝试向后拼接
            if len(a) < MIN_ANSWER_LEN:
                merged_from = []
                appended = 0
                j = i + 1
                while j < len(arr) and appended < MAX_APPEND:
                    nxt = arr[j]
                    nq = nxt.get("question","")
                    na = nxt.get("answer","")
                    nxt_ts = parse_ts(nxt.get("q_ts","") or "")

                    # 时间间隔太大就不拼
                    if cur_ts and nxt_ts:
                        gap = (nxt_ts - cur_ts).total_seconds()
                        if gap > MAX_GAP_SEC:
                            break

                    # 遇到“明显新问题”就停止拼接
                    if is_new_question(nq):
                        break

                    # 拼接 nxt 的 answer（只拼有内容的）
                    if na and na.strip():
                        a = (a + "\n" + na).strip() if a else na.strip()
                        merged_from.append(nxt.get("hash",""))
                        appended += 1

                    j += 1

                if appended > 0:
                    stats["merged_records"] += 1
                    stats["merged_appended_total"] += appended
                    if examples_written < 60 and random.random() < 0.02:
                        ex.write(json.dumps({
                            "source_file": sf,
                            "question": q,
                            "before_answer_len": len(cur.get("answer","")),
                            "after_answer_len": len(a),
                            "appended": appended,
                            "appended_hashes": merged_from,
                            "before_answer": cur.get("answer",""),
                            "after_answer": a,
                        }, ensure_ascii=False) + "\n")
                        examples_written += 1

                    cur["answer"] = clean_text(a)

            # 输出当前记录
            out.write(json.dumps(cur, ensure_ascii=False) + "\n")
            stats["out_total"] += 1
            if len(cur.get("answer","")) < MIN_ANSWER_LEN:
                stats["still_short_after_merge"] += 1

            i += 1  # 这里不跳过被拼接的记录（保守策略：不删除它们，避免误删信息）

In [22]:
OUT_STATS.write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")
print("完成：", stats)
print("输出：", OUT_KB)
print("示例对比：", OUT_EX)

完成： Counter({'in_total': 24485, 'out_total': 24485})
输出： D:\shixi_agent\黄剑企业微信导出\2out5\kb_merged.jsonl
示例对比： D:\shixi_agent\黄剑企业微信导出\2out5\merge_examples.jsonl


## 并入已有知识库（本批跑完后）

1. 把上面各轮里的 `IN_DIR` / 输出路径改成**本批新数据**对应目录，跑完得到新的 `2out4/kb_clean.jsonl`。
2. 在项目目录执行合并（路径按你本机修改）：

```powershell
cd C:\Users\10409\Documents\work\shixi_agent\medical_agent_demo
py -3 -m scripts.kb_merge_jsonl --base "D:\shixi_agent\黄剑企业微信导出\out4\kb_clean.jsonl" --add "D:\shixi_agent\黄剑企业微信导出\本批\2out4\kb_clean.jsonl" --out "D:\shixi_agent\黄剑企业微信导出\out4\kb_clean_merged.jsonl"
```

3. 把 `.env` 里 `KB_SOURCE_JSONL` 指到合并后的文件（或替换原 `kb_clean.jsonl`）。
4. 增量建库：

```powershell
py -3 -m scripts.kb_build_chroma
```

详见仓库内 `docs/kb_merge_append.md`。